# Topic 9 — AdaBoost (Adaptive Boosting)

> **Goal of this notebook:** understand **boosting** — building a strong classifier by training weak learners *sequentially*, each focusing on the mistakes of the last. We'll derive the weight-update rule and implement AdaBoost from scratch with decision stumps.

**Contents**
1. Concept & intuition
2. The mathematics (sample weights, learner weight α, updates)
3. Key hyperparameters
4. The dataset: Breast Cancer
5. Training & the effect of more learners
6. Evaluation
7. AdaBoost from scratch (decision stumps)
8. Pros, cons & when to use
9. Summary

## 1. Concept & Intuition

Where Random Forest builds many trees **in parallel** and averages them, **boosting** builds them **one after another**, each new learner concentrating on the examples the previous ones got wrong.

AdaBoost uses tiny trees called **decision stumps** (depth-1 "weak learners", barely better than guessing). After each stump it:
1. **increases the weight** of misclassified samples (so the next stump pays them more attention),
2. gives the stump a **say (α)** proportional to how accurate it was.

The final prediction is a **weighted vote** of all stumps. Many weak learners combine into one strong one.

## 2. The Mathematics

Labels are $y_i \in \{-1, +1\}$. Start with equal sample weights $w_i = 1/m$. For each round $t$:

1. Train a weak learner $h_t$ using the weights, and compute its weighted error:
$$\varepsilon_t = \sum_i w_i \,\mathbb{1}[h_t(x_i) \neq y_i]$$
2. Compute its **amount of say**:
$$\alpha_t = \tfrac{1}{2} \ln\!\left(\frac{1 - \varepsilon_t}{\varepsilon_t}\right)$$
(accurate learner → large positive α; coin-flip learner → α ≈ 0)
3. **Update sample weights** and renormalise:
$$w_i \leftarrow w_i \, e^{-\alpha_t y_i h_t(x_i)}$$
(weight grows for misclassified points, shrinks for correct ones)

**Final prediction:** $\;H(x) = \text{sign}\!\left(\sum_t \alpha_t h_t(x)\right)$

## 3. Key Hyperparameters

| Parameter | Effect |
|-----------|--------|
| **n_estimators** | number of weak learners (boosting rounds). |
| **learning_rate** | shrinks each learner's contribution; trades off with n_estimators. |
| **estimator** | the weak learner (default: depth-1 tree / stump). |

Unlike Random Forest, **too many rounds *can* overfit**, especially on noisy data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Dataset: Breast Cancer

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. Training & the Effect of More Learners

In [ ]:
ada = AdaBoostClassifier(n_estimators=200, learning_rate=1.0, random_state=42)
ada.fit(X_train, y_train)
print('Test accuracy:', round(ada.score(X_test, y_test), 4))

# staged_score lets us watch accuracy improve round by round
stages = list(ada.staged_score(X_test, y_test))
plt.plot(range(1, len(stages) + 1), stages)
plt.xlabel('number of weak learners'); plt.ylabel('test accuracy')
plt.title('AdaBoost: accuracy as boosting rounds accumulate'); plt.show()

## 6. Evaluation

In [ ]:
y_pred = ada.predict(X_test)
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 7. AdaBoost From Scratch (Decision Stumps)

Implementing the SAMME weight-update loop with depth-1 trees.

In [ ]:
class AdaBoostScratch:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
    def fit(self, X, y):
        # convert labels to {-1, +1}
        y = np.where(y == 0, -1, 1)
        m = len(y)
        w = np.full(m, 1 / m)
        self.stumps, self.alphas = [], []
        for _ in range(self.n_estimators):
            stump = DecisionTreeClassifier(max_depth=1)
            stump.fit(X, y, sample_weight=w)
            pred = stump.predict(X)
            err = w[pred != y].sum() / w.sum()
            err = np.clip(err, 1e-10, 1 - 1e-10)
            alpha = 0.5 * np.log((1 - err) / err)
            w *= np.exp(-alpha * y * pred)
            w /= w.sum()
            self.stumps.append(stump); self.alphas.append(alpha)
        return self
    def predict(self, X):
        agg = sum(a * s.predict(X) for a, s in zip(self.alphas, self.stumps))
        return np.where(np.sign(agg) == -1, 0, 1)

scratch = AdaBoostScratch(n_estimators=200).fit(X_train, y_train)
print('From-scratch accuracy:', round(accuracy_score(y_test, scratch.predict(X_test)), 4))
print('scikit-learn accuracy:', round(accuracy_score(y_test, y_pred), 4))

## 8. Pros, Cons & When to Use

**Pros**
- Turns weak learners into a strong, accurate model.
- Few hyperparameters; little need for scaling.
- Naturally highlights hard examples.

**Cons**
- **Sensitive to noisy data and outliers** (it keeps up-weighting them).
- Sequential → cannot be parallelised like a forest.
- Can overfit with too many rounds.

**When to use**
- Clean tabular data where you want strong accuracy from simple base learners.
- As a lighter alternative to gradient boosting.

## 9. Summary

- AdaBoost trains weak learners **sequentially**, re-weighting misclassified samples each round.
- Each learner's vote weight $\alpha$ grows with its accuracy; the final model is their **weighted vote**.
- On Breast Cancer, accuracy climbed as rounds accumulated.
- Our from-scratch stump-based AdaBoost matched scikit-learn, confirming the weight-update mechanics.